# Desiging Data Flows Betweeen HRM and IAM (AD/LDAP/IAM softwares)

At the core source --> control --> target model
* HRM --> IAM (authoritative flow)
  - Employee lifecycle events (hire, trasfer, terminated)
  - Attributes: role, department, manager, location
* IAM -> Target Systems
  - Provisioning/ Deprovisiong the accounts
  - Role/ entitlement assignment
* Feedback loops (optional but good have it because its very valuable)
  - Target system --> IAM (status, error)
  - IAM -> HRM (rare; usually avoided to preserve HR as source of truth)

Best practices:
* Use even-based triggers instead of batch where it is possible
* Normalize the HR data into identity attributes which can be usable by IAM
* Maintain the **Unique Identifier** (e.g., employee id) across system

---
# Event Sequencing and Dependecy Handling 

Automation must respect order and dependecy (workflows), it will break

**Example: New Hire Flow**
1. HR create employee record
2. Identity is create in the IAM (AD/LDAP)
3. Core System provisioned (email id, directory account)
4. Downstream apps assigned with access

**Key Design Principles:**
1. Workflow engines or orchestration layers ==> managing the sequence
2. Define Hard Dependecies (eg., no app account before directory id exists)
3. add failure handling mechanism -> try catch blocks
4. retries if the activity does not get completed in one attempt
5. esure idempotency (same event can safely re-run)

---
# Exception Scenarios
1. Backdated changes
   * Example: termination entered late in the HRM
   * Risk: unauthorized access persists
   **Handling:**
   * Detect the effective date vs entry date mismatch
   * Trigger immediate corrective actions/ workflows
   * Log and audit retroactive changes
     
2. Emergency Access
   * Temporary elevated access (eg, production outages)
   
   **Handling**
   * Implement time-bound access (auto-expiry)
   * approval requiment & justification workflow/ documentation
   * enforcing audit logging and session monitoring
     
3. Manually Overrides
   * Admins bypass automation (eg, urgent fixes, avoid SLA breaches, some emergency situations)
     
   **Handling**
   * Allow but track as expections
   * periodically reconcile with IAM policies
   * Auto revoke if not aligned with HR driven rules
     
4. Handling the Rehires and Internal Transfers

   **Rehires:**
   * Challenge: reuse vs recreate identity
     
   **Best Practices:**
   * Reassociate with existing identity (if possible)
   * Restoring baseline access, not full historical access
   * Validate the risk before reactivating old entitlements
   * deactive and reactivate workflows can be executed to reconfigure the access according the new job responsibilites
     
   **Internal Transfers:**
   * Triggered by role/department change
     
   **Key actions**
   * Remove old access first, then grant the new access
   * use role-based or attribute-based models
   * avoid access stacking from previous roles